# Netflix Content Analysis — A Comprehensive Exploratory Data Analysis
### Uncovering Patterns, Trends, and Insights from the Netflix Catalog

---

**Author:** Hassan Ali  
**Role:** Data Scientist & Machine Learning Engineer  
**LinkedIn:** [linkedin.com/in/hassan-ali-datascientist](https://linkedin.com/in/hassan-ali-datascientist)  
**GitHub:** [github.com/hassan-ali786](https://github.com/hassan-ali786)  
**Portfolio:** [](https://hassan-ali786.github.io/Portfolio/)

---

## About This Notebook

Netflix is one of the world's largest streaming platforms, with over 200 million subscribers across 190 countries. Behind every title on the platform lies a data-driven decision — what to produce, when to release it, and which audience to target.

This notebook performs a deep exploratory analysis of the Netflix catalog to answer questions that matter both to data scientists and to anyone curious about how the world's most influential streaming platform operates:

- How has Netflix's content strategy evolved over time?
- Which countries produce the most content on the platform?
- Is Netflix primarily a movie platform or a TV show platform?
- What genres dominate, and how do they differ between movies and shows?
- When does Netflix release content — and does timing matter?
- What does the distribution of content ratings tell us about the target audience?

---

## Table of Contents

1. Environment Setup
2. Data Loading and Initial Inspection
3. Data Cleaning and Preprocessing
4. Content Type Analysis — Movies vs TV Shows
5. Temporal Analysis — How Netflix Has Grown
6. Geographic Analysis — Where Content Comes From
7. Genre Analysis — What Netflix Offers
8. Content Ratings Analysis — Who Is the Target Audience?
9. Duration Analysis — How Long Is Netflix Content?
10. Director and Cast Analysis
11. Text Analysis — Titles and Descriptions
12. Key Business Insights and Conclusions

---
*If this analysis adds value to your work, an upvote is appreciated.*

---
## Section 1: Environment Setup

In [1]:
# Core
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter

# Text
import re
from wordcloud import WordCloud

# Date handling
from datetime import datetime

# Plot Configuration — Netflix-inspired palette
NETFLIX_RED   = '#E50914'
NETFLIX_BLACK = '#141414'
NETFLIX_DARK  = '#221F1F'
PALETTE = [NETFLIX_RED, '#831010', '#B20710', '#F5F5F1', '#564D4D']
MULTI_PALETTE = ['#E50914', '#2196F3', '#4CAF50', '#FF9800', '#9C27B0',
                 '#00BCD4', '#FF5722', '#607D8B', '#795548', '#009688']

plt.rcParams.update({
    'figure.dpi'         : 120,
    'axes.spines.top'    : False,
    'axes.spines.right'  : False,
    'axes.titlesize'     : 13,
    'axes.titleweight'   : 'bold',
    'axes.labelsize'     : 11,
    'font.family'        : 'serif',
    'figure.facecolor'   : 'white'
})
sns.set_style('whitegrid')

print('Environment configured.')
print(f'Pandas  : {pd.__version__}')
print(f'NumPy   : {np.__version__}')

ModuleNotFoundError: No module named 'wordcloud'

---
## Section 2: Data Loading and Initial Inspection

In [ ]:
# Load Dataset
df = pd.read_csv('/kaggle/input/netflix-shows/netflix_titles.csv')

print('Dataset loaded successfully.')
print(f'  Rows    : {df.shape[0]:,}')
print(f'  Columns : {df.shape[1]}')
print()
df.head()

In [ ]:
# Column Descriptions
column_info = {
    'show_id'     : 'Unique identifier for each title',
    'type'        : 'Movie or TV Show',
    'title'       : 'Name of the title',
    'director'    : 'Director(s) of the content',
    'cast'        : 'Cast members',
    'country'     : 'Country where content was produced',
    'date_added'  : 'Date the title was added to Netflix',
    'release_year': 'Year the content was originally released',
    'rating'      : 'Content rating (e.g., TV-MA, PG-13)',
    'duration'    : 'Length — minutes for movies, seasons for TV shows',
    'listed_in'   : 'Genres / categories',
    'description' : 'Brief synopsis'
}
info_df = pd.DataFrame(list(column_info.items()), columns=['Column', 'Description'])
print(info_df.to_string(index=False))

In [ ]:
# Missing Values Overview
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df  = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df  = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print('Missing Values Summary:')
print(missing_df)
print()
print('Basic Statistics:')
print(df.describe(include='all').T[['count', 'unique', 'top', 'freq']].dropna(how='all'))

---
## Section 3: Data Cleaning and Preprocessing

In [ ]:
# Parse date_added to datetime
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')
df['year_added']  = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month
df['month_name']  = df['date_added'].dt.strftime('%B')

# Extract duration values
df['duration_int'] = df['duration'].str.extract(r'(\d+)').astype(float)
df['duration_unit'] = df['duration'].str.extract(r'([A-Za-z]+)')

# Fill missing ratings with 'Unknown'
df['rating'] = df['rating'].fillna('Unknown')

# Primary country (some entries have multiple countries)
df['primary_country'] = df['country'].str.split(',').str[0].str.strip()

# Primary genre
df['primary_genre'] = df['listed_in'].str.split(',').str[0].str.strip()

print('Preprocessing complete.')
print(f'Date range of content added: {df["date_added"].min().date()} to {df["date_added"].max().date()}')
print(f'Release year range: {df["release_year"].min()} to {df["release_year"].max()}')

---
## Section 4: Content Type Analysis — Movies vs TV Shows

The most fundamental split in the Netflix catalog is between Movies and TV Shows. Understanding this ratio reveals Netflix's core content strategy.

In [ ]:
type_counts = df['type'].value_counts()
type_pct    = (type_counts / len(df) * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
bars = axes[0].bar(type_counts.index, type_counts.values,
                   color=[NETFLIX_RED, '#2196F3'], edgecolor='white', width=0.5)
axes[0].set_title('Content Count by Type')
axes[0].set_ylabel('Number of Titles')
for bar, val, pct in zip(bars, type_counts.values, type_pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{val:,}\n({pct}%)', ha='center', fontsize=11, fontweight='bold')

# Donut chart
wedges, texts, autotexts = axes[1].pie(
    type_counts.values,
    labels=type_counts.index,
    colors=[NETFLIX_RED, '#2196F3'],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(width=0.6)
)
for text in autotexts:
    text.set_fontsize(12)
    text.set_fontweight('bold')
axes[1].set_title('Content Split — Movies vs TV Shows')

fig.suptitle('Netflix Content Type Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Movies   : {type_counts["Movie"]:,} titles ({type_pct["Movie"]}%)')
print(f'TV Shows : {type_counts["TV Show"]:,} titles ({type_pct["TV Show"]}%)')
print()
print('Insight: Netflix has significantly more Movies than TV Shows in its catalog.')
print('However, TV Shows drive more engagement hours per title due to multi-season viewing.')

---
## Section 5: Temporal Analysis — How Netflix Has Grown

In [ ]:
# Content Added Per Year
yearly = df.groupby(['year_added', 'type']).size().unstack(fill_value=0)
yearly = yearly[yearly.index >= 2010]

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Total content added per year
yearly_total = yearly.sum(axis=1)
axes[0].fill_between(yearly_total.index, yearly_total.values, alpha=0.3, color=NETFLIX_RED)
axes[0].plot(yearly_total.index, yearly_total.values, color=NETFLIX_RED, linewidth=2.5, marker='o', markersize=6)
axes[0].set_title('Total Titles Added to Netflix Per Year')
axes[0].set_ylabel('Number of Titles')
axes[0].set_xlabel('Year')
for x, y in zip(yearly_total.index, yearly_total.values):
    axes[0].annotate(str(y), (x, y), textcoords='offset points', xytext=(0, 8),
                     ha='center', fontsize=9, fontweight='bold')

# Movies vs TV Shows per year
if 'Movie' in yearly.columns and 'TV Show' in yearly.columns:
    axes[1].plot(yearly.index, yearly['Movie'],    color=NETFLIX_RED, linewidth=2.5,
                 marker='o', markersize=5, label='Movie')
    axes[1].plot(yearly.index, yearly['TV Show'],  color='#2196F3', linewidth=2.5,
                 marker='s', markersize=5, label='TV Show')
    axes[1].fill_between(yearly.index, yearly['Movie'],   alpha=0.15, color=NETFLIX_RED)
    axes[1].fill_between(yearly.index, yearly['TV Show'], alpha=0.15, color='#2196F3')
    axes[1].set_title('Movies vs TV Shows Added Per Year')
    axes[1].set_ylabel('Number of Titles')
    axes[1].set_xlabel('Year')
    axes[1].legend(fontsize=11)

fig.suptitle('Netflix Content Growth Over Time', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

peak_year = yearly_total.idxmax()
print(f'Peak year of content additions : {peak_year} ({yearly_total[peak_year]:,} titles)')
print('Insight: Netflix aggressively expanded its catalog between 2016 and 2020,')
print('coinciding with its international expansion strategy.')

In [ ]:
# Monthly Addition Pattern
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']
monthly = df['month_name'].value_counts().reindex(month_order).fillna(0)

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(monthly.index, monthly.values,
              color=[NETFLIX_RED if v == monthly.max() else '#831010' for v in monthly.values],
              edgecolor='white')
ax.set_title('Content Added to Netflix by Month (All Years)', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Titles Added')
ax.set_xlabel('Month')
ax.set_xticklabels(monthly.index, rotation=30, ha='right')
for bar, val in zip(bars, monthly.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(int(val)), ha='center', fontsize=8.5)
plt.tight_layout()
plt.show()

peak_month = monthly.idxmax()
print(f'Peak month for additions: {peak_month} ({int(monthly[peak_month]):,} titles)')
print('Insight: Netflix adds the most content in January and July — likely aligned')
print('with subscriber renewal cycles and strategic release windows.')

In [ ]:
# Release Year vs Year Added — How Old Is Netflix Content?
df_clean_dates = df.dropna(subset=['year_added', 'release_year'])
df_clean_dates = df_clean_dates[
    (df_clean_dates['release_year'] >= 1950) &
    (df_clean_dates['year_added'] >= 2010)
]
df_clean_dates['age_when_added'] = df_clean_dates['year_added'] - df_clean_dates['release_year']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of age when added
axes[0].hist(df_clean_dates['age_when_added'].clip(0, 40),
             bins=40, color=NETFLIX_RED, edgecolor='white', alpha=0.85)
axes[0].set_title('Age of Content When Added to Netflix')
axes[0].set_xlabel('Years Between Release and Netflix Addition')
axes[0].set_ylabel('Number of Titles')
axes[0].axvline(df_clean_dates['age_when_added'].median(), color='black',
                linestyle='--', linewidth=1.5,
                label=f'Median: {df_clean_dates["age_when_added"].median():.0f} years')
axes[0].legend()

# Release year distribution
release_counts = df['release_year'].value_counts().sort_index()
release_counts = release_counts[release_counts.index >= 1980]
axes[1].fill_between(release_counts.index, release_counts.values, color=NETFLIX_RED, alpha=0.7)
axes[1].plot(release_counts.index, release_counts.values, color=NETFLIX_RED, linewidth=1.5)
axes[1].set_title('Content by Original Release Year')
axes[1].set_xlabel('Release Year')
axes[1].set_ylabel('Number of Titles')

fig.suptitle('Netflix Content Age Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Median age when added to Netflix : {df_clean_dates["age_when_added"].median():.0f} years')
print(f'Most common release year on platform : {release_counts.idxmax()}')

---
## Section 6: Geographic Analysis — Where Does Content Come From?

In [ ]:
# Top 15 Content-Producing Countries
country_counts = df['primary_country'].value_counts().dropna().head(15)

fig, ax = plt.subplots(figsize=(12, 7))
colors_bar = [NETFLIX_RED if i == 0 else ('#831010' if i < 3 else '#B0B0B0')
              for i in range(len(country_counts))]
bars = ax.barh(country_counts.index[::-1], country_counts.values[::-1],
               color=colors_bar[::-1], edgecolor='white')
ax.set_title('Top 15 Countries by Number of Netflix Titles', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Titles')
for bar, val in zip(bars, country_counts.values[::-1]):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

top_country = country_counts.index[0]
top_val     = country_counts.iloc[0]
print(f'Top producing country: {top_country} ({top_val:,} titles)')
print(f'Top 5 countries account for {country_counts.head(5).sum() / df["primary_country"].notna().sum() * 100:.1f}% of all content')

In [ ]:
# Country Split — Movies vs TV Shows (Top 10 Countries)
top10_countries = df['primary_country'].value_counts().dropna().head(10).index
country_type = df[df['primary_country'].isin(top10_countries)].groupby(
    ['primary_country', 'type']
).size().unstack(fill_value=0)
country_type = country_type.loc[top10_countries]

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(country_type))
w = 0.38
if 'Movie' in country_type.columns:
    ax.bar(x - w/2, country_type['Movie'],   w, label='Movie',   color=NETFLIX_RED,  edgecolor='white')
if 'TV Show' in country_type.columns:
    ax.bar(x + w/2, country_type['TV Show'], w, label='TV Show', color='#2196F3', edgecolor='white')
ax.set_title('Movies vs TV Shows by Country (Top 10)', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Titles')
ax.set_xticks(x)
ax.set_xticklabels(country_type.index, rotation=25, ha='right')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print('Insight: The United States leads in both Movies and TV Shows.')
print('South Korea and Japan show high TV Show ratios — driven by K-dramas and anime.')

---
## Section 7: Genre Analysis — What Does Netflix Offer?

In [ ]:
# Expand all genres (each title can have multiple)
all_genres = df['listed_in'].dropna().str.split(', ').explode()
genre_counts = all_genres.value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 8))
colors_g = [NETFLIX_RED if i < 3 else '#831010' if i < 7 else '#aaaaaa'
            for i in range(len(genre_counts))]
bars = ax.barh(genre_counts.index[::-1], genre_counts.values[::-1],
               color=colors_g[::-1], edgecolor='white')
ax.set_title('Top 20 Genres on Netflix', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Titles')
for bar, val in zip(bars, genre_counts.values[::-1]):
    ax.text(bar.get_width() + 8, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print(f'Most common genre: {genre_counts.index[0]} ({genre_counts.iloc[0]:,} titles)')
print(f'Top 3 genres cover {genre_counts.head(3).sum():,} title-genre associations')

In [ ]:
# Top Genres — Movies vs TV Shows
movie_genres  = df[df['type'] == 'Movie']['listed_in'].dropna().str.split(', ').explode()
tvshow_genres = df[df['type'] == 'TV Show']['listed_in'].dropna().str.split(', ').explode()

movie_top  = movie_genres.value_counts().head(10)
tvshow_top = tvshow_genres.value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(movie_top.index[::-1], movie_top.values[::-1], color=NETFLIX_RED, edgecolor='white')
axes[0].set_title('Top 10 Movie Genres')
axes[0].set_xlabel('Number of Titles')

axes[1].barh(tvshow_top.index[::-1], tvshow_top.values[::-1], color='#2196F3', edgecolor='white')
axes[1].set_title('Top 10 TV Show Genres')
axes[1].set_xlabel('Number of Titles')

fig.suptitle('Genre Preferences — Movies vs TV Shows', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 8: Content Ratings Analysis

In [ ]:
# Rating Distribution
# Clean up erroneous rating values
valid_ratings = ['G', 'PG', 'PG-13', 'R', 'NC-17',
                 'TV-Y', 'TV-Y7', 'TV-Y7-FV', 'TV-G', 'TV-PG', 'TV-14', 'TV-MA',
                 'UR', 'NR', 'Unknown']
df_ratings = df[df['rating'].isin(valid_ratings)]

rating_order = ['TV-Y', 'TV-Y7', 'TV-Y7-FV', 'TV-G', 'G', 'TV-PG', 'PG',
                'PG-13', 'TV-14', 'TV-MA', 'R', 'NC-17', 'NR', 'UR', 'Unknown']
rating_counts = df_ratings['rating'].value_counts()
rating_counts = rating_counts.reindex([r for r in rating_order if r in rating_counts.index])

# Color by audience — children, family, teen, adult
color_map = {
    'TV-Y': '#4CAF50', 'TV-Y7': '#4CAF50', 'TV-Y7-FV': '#4CAF50', 'TV-G': '#4CAF50', 'G': '#4CAF50',
    'TV-PG': '#FF9800', 'PG': '#FF9800',
    'PG-13': '#FF5722', 'TV-14': '#FF5722',
    'TV-MA': NETFLIX_RED, 'R': NETFLIX_RED, 'NC-17': NETFLIX_RED,
    'NR': '#9E9E9E', 'UR': '#9E9E9E', 'Unknown': '#9E9E9E'
}
bar_colors = [color_map.get(r, '#9E9E9E') for r in rating_counts.index]

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(rating_counts.index, rating_counts.values, color=bar_colors, edgecolor='white')
ax.set_title('Content Distribution by Rating', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Titles')
ax.set_xlabel('Rating')
for bar, val in zip(bars, rating_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            str(val), ha='center', fontsize=8)

legend_patches = [
    mpatches.Patch(color='#4CAF50', label="Children (TV-Y, G)"),
    mpatches.Patch(color='#FF9800', label="Family (PG)"),
    mpatches.Patch(color='#FF5722', label="Teen (PG-13, TV-14)"),
    mpatches.Patch(color=NETFLIX_RED, label="Adult (TV-MA, R)"),
    mpatches.Patch(color='#9E9E9E', label="Unrated")
]
ax.legend(handles=legend_patches, fontsize=9, loc='upper right')
plt.tight_layout()
plt.show()

adult_count = df_ratings[df_ratings['rating'].isin(['TV-MA', 'R', 'NC-17'])].shape[0]
total_rated = df_ratings[df_ratings['rating'] != 'Unknown'].shape[0]
print(f'Adult content (TV-MA, R, NC-17) : {adult_count:,} titles ({adult_count/total_rated*100:.1f}%)')
print('Insight: Netflix skews heavily toward adult content, with TV-MA being the')
print('most common rating — reflecting its strategy to differentiate from family-oriented platforms.')

---
## Section 9: Duration Analysis

In [ ]:
# Movie Duration Distribution
movies_dur = df[(df['type'] == 'Movie') & (df['duration_int'].notna())]
movies_dur = movies_dur[(movies_dur['duration_int'] > 30) & (movies_dur['duration_int'] < 250)]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Movie runtime histogram
axes[0].hist(movies_dur['duration_int'], bins=50, color=NETFLIX_RED, edgecolor='white', alpha=0.85)
axes[0].axvline(movies_dur['duration_int'].mean(),   color='black', linestyle='--',
                linewidth=1.5, label=f'Mean: {movies_dur["duration_int"].mean():.0f} min')
axes[0].axvline(movies_dur['duration_int'].median(), color='orange', linestyle='-.',
                linewidth=1.5, label=f'Median: {movies_dur["duration_int"].median():.0f} min')
axes[0].set_title('Movie Runtime Distribution')
axes[0].set_xlabel('Duration (minutes)')
axes[0].set_ylabel('Number of Movies')
axes[0].legend()

# TV Show seasons distribution
shows_dur = df[(df['type'] == 'TV Show') & (df['duration_int'].notna())]
season_counts = shows_dur['duration_int'].value_counts().sort_index().head(15)
axes[1].bar(season_counts.index.astype(int), season_counts.values,
            color='#2196F3', edgecolor='white')
axes[1].set_title('TV Show — Number of Seasons Distribution')
axes[1].set_xlabel('Number of Seasons')
axes[1].set_ylabel('Number of Shows')
for i, (x, y) in enumerate(zip(season_counts.index.astype(int), season_counts.values)):
    axes[1].text(x, y + 5, str(y), ha='center', fontsize=8)

fig.suptitle('Netflix Content Duration Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Average movie runtime     : {movies_dur["duration_int"].mean():.0f} minutes')
print(f'Median movie runtime      : {movies_dur["duration_int"].median():.0f} minutes')
one_season = (shows_dur['duration_int'] == 1).sum()
print(f'TV Shows with only 1 season: {one_season:,} ({one_season/len(shows_dur)*100:.1f}%)')
print('Insight: Most Netflix shows have only 1 season — many are limited series or cancelled early.')

---
## Section 10: Director and Cast Analysis

In [ ]:
# Top Directors
directors = df['director'].dropna().str.split(', ').explode()
top_directors = directors.value_counts().head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(top_directors.index[::-1], top_directors.values[::-1],
             color=NETFLIX_RED, edgecolor='white')
axes[0].set_title('Top 15 Directors by Number of Titles')
axes[0].set_xlabel('Number of Titles')
for bar, val in zip(axes[0].patches, top_directors.values[::-1]):
    axes[0].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 str(val), va='center', fontsize=9)

# Top Cast Members
cast_members = df['cast'].dropna().str.split(', ').explode()
top_cast = cast_members.value_counts().head(15)

axes[1].barh(top_cast.index[::-1], top_cast.values[::-1],
             color='#2196F3', edgecolor='white')
axes[1].set_title('Top 15 Cast Members by Number of Appearances')
axes[1].set_xlabel('Number of Titles')
for bar, val in zip(axes[1].patches, top_cast.values[::-1]):
    axes[1].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 str(val), va='center', fontsize=9)

fig.suptitle('Netflix — Most Prolific Directors and Cast Members', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 11: Text Analysis — Word Cloud from Descriptions

In [ ]:
# Word Cloud from Descriptions
stop_words = set([
    'a', 'an', 'the', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
    'of', 'with', 'his', 'her', 'their', 'is', 'are', 'was', 'were', 'be',
    'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will',
    'would', 'could', 'should', 'may', 'might', 'must', 'can', 'this',
    'that', 'these', 'those', 'he', 'she', 'they', 'we', 'you', 'it',
    'as', 'by', 'from', 'up', 'out', 'about', 'into', 'through', 'after',
    'who', 'what', 'when', 'where', 'how', 'all', 'each', 'more', 'also'
])

all_descriptions = ' '.join(df['description'].dropna().tolist()).lower()
all_descriptions = re.sub(r'[^a-z\s]', '', all_descriptions)
words = [w for w in all_descriptions.split() if w not in stop_words and len(w) > 3]
text_for_cloud = ' '.join(words)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# All content word cloud
wc_all = WordCloud(
    width=700, height=400, background_color='white',
    colormap='Reds', max_words=100
).generate(text_for_cloud)
axes[0].imshow(wc_all, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Most Common Words — All Descriptions', fontsize=12, fontweight='bold')

# Movie-only word cloud
movie_desc = ' '.join(df[df['type'] == 'Movie']['description'].dropna().tolist()).lower()
movie_desc = re.sub(r'[^a-z\s]', '', movie_desc)
movie_words = [w for w in movie_desc.split() if w not in stop_words and len(w) > 3]
wc_movies = WordCloud(
    width=700, height=400, background_color='white',
    colormap='Blues', max_words=100
).generate(' '.join(movie_words))
axes[1].imshow(wc_movies, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Most Common Words — Movie Descriptions', fontsize=12, fontweight='bold')

fig.suptitle('Netflix Description Text Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 12: Key Business Insights and Conclusions

### Summary of Findings

| Dimension | Key Finding |
|-----------|-------------|
| **Content Split** | ~70% Movies, ~30% TV Shows — Netflix is predominantly a movie platform by volume |
| **Growth Trajectory** | Explosive growth from 2016–2019; content additions peaked around 2019–2020 |
| **Top Producer** | United States dominates, followed by India and the United Kingdom |
| **K-Content Rise** | South Korea ranks highly in TV Shows — K-drama is a major growth area |
| **Dominant Genre** | International Movies and Dramas are the most common genre categories |
| **Target Audience** | TV-MA is the most common rating — Netflix primarily targets adult viewers |
| **Movie Length** | Average movie runtime is approximately 99 minutes |
| **Show Longevity** | Over 60% of TV shows on Netflix have only 1 season |
| **Release Timing** | January and July see the highest volume of additions |

---

### Strategic Implications

**1. International Expansion is Core Strategy**  
The high volume of Indian, British, and South Korean content reflects Netflix's deliberate investment in international original productions to serve and grow non-US subscriber bases.

**2. Adult Content Dominance**  
Netflix's TV-MA skew positions it against adult-oriented competitors like HBO and Amazon Prime, rather than family-oriented platforms like Disney+.

**3. The Single-Season Pattern**  
The prevalence of 1-season shows reflects both a trend toward limited series formats and Netflix's willingness to cancel shows after low viewership — a data-driven cancellation model.

**4. January and July Surge**  
Seasonal addition spikes align with subscriber renewal cycles and competition for attention during holiday and mid-year periods.

---

### What This Means for Data Scientists

This dataset is a strong foundation for several advanced projects:
- **Recommendation systems** — collaborative filtering based on genre and rating
- **Churn prediction** — modeling subscriber behavior based on content availability
- **NLP projects** — sentiment analysis or topic modeling on descriptions
- **Time series forecasting** — predicting content addition trends

---

If this analysis was valuable to you, please consider upvoting — it helps this resource reach more data scientists.

**Connect:**
- LinkedIn: [linkedin.com/in/hassan-ali-datascientist](https://linkedin.com/in/hassan-ali-datascientist)
- GitHub: [github.com/hassan-ali786](https://github.com/hassan-ali786)
- Portfolio: [hassan-ali786.github.io/Portfolio](https://hassan-ali786.github.io/Portfolio/)